In [ ]:
# 🎙️ Egyptian Arabic Video Transcription
### Google Colab — Refactored & Dependency-Fixed

**Pipeline:**
1. Install all dependencies (correct order)
2. Mount Google Drive & load Seamless M4T v2
3. Upload video → extract audio
4. Transcribe with SeamlessM4T (chunked)
5. Align word-level timestamps with WhisperX
6. Compare transcript vs original script

## 🔧 Step 1 — Install All Dependencies
> Run this cell **once** and then **restart the runtime** when prompted.
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTANT: After this cell finishes, click:
#   Runtime → Restart runtime
# Then run from Step 2 onwards. Do NOT re-run this cell.
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))

def uninstall(*pkgs):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "uninstall", "-y"] + list(pkgs),
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

import torch
print(f"Detected torch {torch.__version__}")

# ── 1. Remove torchvision (torch 2.8 nms conflict, not needed for audio) ─────
print("[1/7] Removing torchvision...")
uninstall("torchvision")

# ── 2. Force-reinstall transformers ──────────────────────────────────────────
print("[2/7] Reinstalling transformers...")
uninstall("transformers")
pip("--force-reinstall", "--no-cache-dir", "transformers")

# ── 3. bitsandbytes + accelerate ─────────────────────────────────────────────
print("[3/7] bitsandbytes, accelerate...")
pip("bitsandbytes>=0.41.0", "accelerate>=0.26.0")

# ── 4. audio / video libraries ───────────────────────────────────────────────
print("[4/7] moviepy, librosa, soundfile...")
pip("moviepy", "librosa", "soundfile")

# ── 5. WhisperX and deps ─────────────────────────────────────────────────────
print("[5/7] faster-whisper, whisperx...")
pip("faster-whisper")
pip("whisperx")

# ── 6. remaining deps ────────────────────────────────────────────────────────
print("[6/7] omegaconf, pandas, openpyxl...")
pip("omegaconf", "pandas", "openpyxl")

# ── 7. Force NumPy + numba LAST — redefine pip here in case kernel restarted ─
print("[7/7] Pinning numpy<2.0 and numba (LAST step)...")
import subprocess, sys
_pip = lambda *a: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(a))
_pip("--force-reinstall", "numpy<2.0")
_pip("numba<0.60")

print()
print("✅ All dependencies installed!")
print("👉  Runtime → Restart runtime  → then continue from Step 2.")
## 📦 Step 2 — Imports
> Run after restarting the runtime.
import torch
import torchaudio
import numpy as np
import os
import gc
import re
import json
import difflib
import pandas as pd
import librosa
import soundfile as sf

from transformers import AutoProcessor, SeamlessM4Tv2Model
from moviepy.editor import VideoFileClip
from IPython.display import Audio, HTML, display
from google.colab import files, drive

# ── Patch torch.load for WhisperX / omegaconf compatibility ──────────────────
import torch.serialization
try:
    from omegaconf.listconfig import ListConfig
    from omegaconf.dictconfig import DictConfig
    torch.serialization.add_safe_globals([ListConfig, DictConfig])
except ImportError:
    pass

_orig_torch_load = torch.load
torch.load = lambda *a, **kw: _orig_torch_load(*a, **{**kw, "weights_only": False})

# ── Device ────────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "float32"

if device == "cuda":
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — transcription will be very slow on CPU.")

## 🤖 Step 3 — Load Seamless M4T v2 Model

The model is ~10 GB. It will be downloaded from HuggingFace once and cached in
your Google Drive under a folder called .

**If you already downloaded it before**, this cell will load it from Drive
without re-downloading.

# ── Mount Drive ───────────────────────────────────────────────────────────────
print("Mounting Google Drive...")
drive.mount("/content/drive", force_remount=False)

# ── Locate or create model folder ─────────────────────────────────────────────
MYDRIVE = "/content/drive/MyDrive"
model_dir = None

for item in os.listdir(MYDRIVE):
    if "seamless" in item.lower():
        candidate = os.path.join(MYDRIVE, item)
        if os.path.isdir(candidate):
            model_dir = candidate
            print(f"Found existing model folder: {model_dir}")
            break

if model_dir is None:
    model_dir = os.path.join(MYDRIVE, "seamless-m4t-v2-large")
    os.makedirs(model_dir, exist_ok=True)
    print(f"Created new model folder: {model_dir}")

# ── Load or download ──────────────────────────────────────────────────────────
config_path = os.path.join(model_dir, "config.json")

if os.path.exists(config_path):
    print("Loading model from Google Drive (no download needed)...")
    local_files_only = True
    model_id = model_dir
else:
    print("Model not found locally — downloading from HuggingFace (~10 GB)...")
    local_files_only = False
    model_id = "facebook/seamless-m4t-v2-large"

processor = AutoProcessor.from_pretrained(model_id, local_files_only=local_files_only)

model = SeamlessM4Tv2Model.from_pretrained(
    model_id,
    load_in_8bit=True,       # ~50% VRAM savings
    device_map="auto",
    local_files_only=local_files_only,
)

if not os.path.exists(config_path):
    print("Saving model to Google Drive for future use...")
    processor.save_pretrained(model_dir)
    model.save_pretrained(model_dir)
    print(f"✅ Saved to: {model_dir}")

print("✅ Model ready!")
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   GPU VRAM used: {used:.1f} / {total:.1f} GB")

## 📹 Step 4 — Upload Your Video
print("Please upload your video file (mp4, avi, mov, mkv...)")
uploaded = files.upload()
video_filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {video_filename}")

print("[7/7] Pinning numpy<2.0 and numba (LAST step)...")
pip("--force-reinstall", "numpy<2.0")
pip("numba<0.60")
## 🔊 Step 5 — Extract Audio from Video
audio_filename = "/content/extracted_audio.wav"

print("Extracting audio...")
video = VideoFileClip(video_filename)
video.audio.write_audiofile(audio_filename, codec="pcm_s16le", verbose=False, logger=None)
video.close()
print(f"✅ Audio saved to: {audio_filename}")

# Load & preview
audio_array, sample_rate = librosa.load(audio_filename, sr=16000)
duration = len(audio_array) / sample_rate
print(f"   Duration : {duration:.1f}s  ({duration/60:.1f} min)")
print(f"   Sample rate: {sample_rate} Hz")

display(Audio(audio_filename, autoplay=False))

## ✍️ Step 6 — Transcribe with SeamlessM4T (Chunked)

Audio is processed in **20-second chunks** to avoid running out of GPU memory.
Egyptian Arabic target language code:

def transcribe_with_chunking(audio_array, chunk_length_seconds=20, overlap_seconds=1):
    """Transcribe long audio by splitting into overlapping chunks."""
    sr = 16000
    chunk_size   = chunk_length_seconds * sr
    overlap_size = overlap_seconds      * sr
    step         = chunk_size - overlap_size

    transcriptions = []
    starts = list(range(0, len(audio_array), step))
    total  = len(starts)

    print(f"Processing {total} chunk(s) of ~{chunk_length_seconds}s each...\n")

    for idx, i in enumerate(starts, 1):
        chunk = audio_array[i : i + chunk_size]

        if len(chunk) < sr * 0.5:   # skip < 0.5 s
            continue

        print(f"  Chunk {idx}/{total}  ({len(chunk)/sr:.1f}s)...", end=" ", flush=True)

        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

            inputs = processor(
                audios=chunk,
                sampling_rate=sr,
                return_tensors="pt",
            ).to(device)

            with torch.no_grad():
                output_tokens = model.generate(
                    **inputs,
                    tgt_lang="arz",
                    generate_speech=False,
                    max_new_tokens=256,
                )

            text = processor.decode(
                output_tokens[0].tolist()[0],
                skip_special_tokens=True,
            )
            transcriptions.append(text)
            print("✓")

            del inputs, output_tokens
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except Exception as e:
            print(f"✗ Error: {e}")

    return " ".join(transcriptions)


print("Starting transcription...\n")
transcription = transcribe_with_chunking(audio_array, chunk_length_seconds=20)

print("\n" + "="*60)
print("TRANSCRIPTION (Egyptian Arabic / arz):")
print("="*60)
print(transcription)
print("="*60)
## ⏱️ Step 7 — Word-Level Alignment with WhisperX

WhisperX aligns the transcription to the audio giving exact timestamps per word.

import whisperx

print("Loading WhisperX model...")
whisperx_model = whisperx.load_model("large-v2", device, compute_type=compute_type)
print(f"✅ WhisperX loaded on {device}")


def align_with_whisperx(audio_array, sample_rate=16000):
    """Return a list of dicts with keys: word, start, end, duration."""
    temp_audio = "/content/temp_align.wav"
    sf.write(temp_audio, audio_array, sample_rate)
    audio = whisperx.load_audio(temp_audio)

    # Step 1 – get segment timestamps
    print("Step 1: Whisper transcription...")
    result = whisperx_model.transcribe(audio, batch_size=16, language="ar")
    print(f"  {len(result['segments'])} segments found.")

    # Step 2 – word-level alignment
    print("Step 2: Word-level alignment...")
    try:
        model_a, metadata = whisperx.load_align_model(language_code="ar", device=device)
        result_aligned = whisperx.align(
            result["segments"], model_a, metadata,
            audio, device, return_char_alignments=False,
        )
        del model_a
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("  ✅ Word-level alignment done.")
    except Exception as e:
        print(f"  ⚠️ Alignment failed ({e}), falling back to segment timestamps.")
        result_aligned = result

    # Extract words
    word_timestamps = []
    for seg in result_aligned["segments"]:
        if "words" in seg and seg["words"]:
            for w in seg["words"]:
                if w.get("word", "").strip():
                    word_timestamps.append({
                        "word"    : w["word"].strip(),
                        "start"   : round(w["start"], 3),
                        "end"     : round(w["end"],   3),
                        "duration": round(w["end"] - w["start"], 3),
                    })
        else:
            words = seg["text"].strip().split()
            dur   = seg["end"] - seg["start"]
            wdur  = dur / max(len(words), 1)
            for k, word in enumerate(words):
                s = seg["start"] + k * wdur
                word_timestamps.append({
                    "word"    : word,
                    "start"   : round(s,        3),
                    "end"     : round(s + wdur, 3),
                    "duration": round(wdur,      3),
                })

    os.remove(temp_audio)
    return word_timestamps


print("\nAligning audio...")
aligned_segments = align_with_whisperx(audio_array)

print(f"\n✅ {len(aligned_segments)} words with timestamps")
print("\nFirst 20 words:")
for i, seg in enumerate(aligned_segments[:20], 1):
    print(f"  {i:3}. [{seg['start']:7.3f}s – {seg['end']:7.3f}s]  {seg['word']}")
if len(aligned_segments) > 20:
    print(f"  ... and {len(aligned_segments)-20} more")
## 📜 Step 8 — Upload Your Original Script

Supported formats: , ,
print("Upload your original script file (.txt / .json / .srt):")
uploaded_script = files.upload()
SCRIPT_FILE = list(uploaded_script.keys())[0]
print(f"✅ Script uploaded: {SCRIPT_FILE}")

## 📊 Step 9 — Compare Script vs Transcript
# ── Arabic normalisation ──────────────────────────────────────────────────────
def normalize_arabic(text):
    text = text.lower()
    text = re.sub(r"[\u0617-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED]", "", text)
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى",      "ي", text)
    text = re.sub("ة",      "ه", text)
    text = re.sub("گ",      "ك", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+",    " ", text)
    return text.strip()

# ── SRT helpers ───────────────────────────────────────────────────────────────
def srt_to_seconds(ts):
    ts = ts.replace(",", ".")
    h, m, s = ts.split(":")
    return int(h)*3600 + int(m)*60 + float(s)

def parse_srt(content):
    segs, lines, i = [], content.strip().split("\n"), 0
    while i < len(lines):
        if lines[i].strip().isdigit():
            i += 1
        if i < len(lines) and "-->" in lines[i]:
            parts = lines[i].split("-->")
            s, e  = srt_to_seconds(parts[0].strip()), srt_to_seconds(parts[1].strip().split()[0])
            i += 1
            texts = []
            while i < len(lines) and lines[i].strip() and not lines[i].strip().isdigit():
                texts.append(lines[i].strip()); i += 1
            if texts:
                segs.append({"text": " ".join(texts), "start": s, "end": e})
        i += 1
    return segs

def load_script(path):
    with open(path, encoding="utf-8") as f:
        content = f.read()
    if path.endswith(".json"):
        data = json.loads(content)
        if isinstance(data, list) and data and "start" in data[0]:
            return data
        return [{"text": data.get("text", content), "start": 0, "end": 0}]
    if path.endswith(".srt"):
        return parse_srt(content)
    return [{"text": content, "start": 0, "end": 0}]

# ── Load & normalize ──────────────────────────────────────────────────────────
script_segments = load_script(SCRIPT_FILE)
script_text     = " ".join(seg["text"].strip() for seg in script_segments)
norm_script     = normalize_arabic(script_text)
norm_transcript = " ".join(normalize_arabic(seg["word"]) for seg in aligned_segments)

print(f"Script    : {len(norm_script.split())} words")
print(f"Transcript: {len(norm_transcript.split())} words")

# ── Word-level comparison ─────────────────────────────────────────────────────
def compare(norm_script, norm_transcript, aligned_segments):
    sw = norm_script.split()
    tw = norm_transcript.split()

    ts_list = [{"word": normalize_arabic(s["word"]), "start": s["start"], "end": s["end"]}
               for s in aligned_segments]

    rows, t_idx = [], 0
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, sw, tw).get_opcodes():
        if tag == "equal":
            for k in range(i2 - i1):
                ts = ts_list[t_idx]["start"] if t_idx < len(ts_list) else None
                rows.append({"Status": "✓ Match", "Script": sw[i1+k], "Transcript": tw[j1+k],
                              "Timestamp": f"{ts:.2f}s" if ts is not None else "N/A", "Note": ""})
                t_idx += 1
        elif tag == "replace":
            for k in range(max(i2-i1, j2-j1)):
                s_w = sw[i1+k] if i1+k < i2 else ""
                t_w = tw[j1+k] if j1+k < j2 else ""
                ts  = ts_list[t_idx]["start"] if t_w and t_idx < len(ts_list) else None
                if s_w and t_w:
                    status, note = "🟡 Changed", f"{s_w!r} -> {t_w!r}"
                elif s_w:
                    status, note, ts = "🔴 Skipped", f"{s_w!r} not said", None
                else:
                    status, note = "🟢 Added", f"{t_w!r} added"
                rows.append({"Status": status, "Script": s_w or "-", "Transcript": t_w or "-",
                              "Timestamp": f"{ts:.2f}s" if ts is not None else "N/A", "Note": note})
                if t_w:
                    t_idx += 1
        elif tag == "delete":
            for k in range(i2-i1):
                rows.append({"Status": "🔴 Skipped", "Script": sw[i1+k], "Transcript": "-",
                              "Timestamp": "N/A", "Note": f"{sw[i1+k]!r} not said"})
        elif tag == "insert":
            for k in range(j2-j1):
                ts = ts_list[t_idx]["start"] if t_idx < len(ts_list) else None
                rows.append({"Status": "🟢 Added", "Script": "-", "Transcript": tw[j1+k],
                              "Timestamp": f"{ts:.2f}s" if ts is not None else "N/A",
                              "Note": f"{tw[j1+k]!r} added"})
                t_idx += 1
    return pd.DataFrame(rows)

df = compare(norm_script, norm_transcript, aligned_segments)

# ── Summary ───────────────────────────────────────────────────────────────────
total   = len(df)
matched = (df["Status"] == "✓ Match").sum()
changed = (df["Status"] == "🟡 Changed").sum()
skipped = (df["Status"] == "🔴 Skipped").sum()
added   = (df["Status"] == "🟢 Added").sum()

print("\n" + "="*60)
print("📊 SUMMARY")
print("="*60)
print(f"Total     : {total}")
print(f"✓ Match   : {matched}  ({matched/total*100:.1f}%)")
print(f"🟡 Changed: {changed}  ({changed/total*100:.1f}%)")
print(f"🔴 Skipped: {skipped}  ({skipped/total*100:.1f}%)")
print(f"🟢 Added  : {added}    ({added/total*100:.1f}%)")
accuracy = matched / max(matched + changed + skipped, 1) * 100
print(f"\n🎯 Word Accuracy: {accuracy:.1f}%")

# ── Display full table ────────────────────────────────────────────────────────
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 60)
print("\n📋 Full Comparison Table:")
display(df)
## 💾 Step 10 — Export Results
# Save to Drive
output_csv   = os.path.join(MYDRIVE, "word_comparison.csv")
output_excel = os.path.join(MYDRIVE, "word_comparison.xlsx")

df.to_csv(output_csv,   index=False, encoding="utf-8-sig")
df.to_excel(output_excel, index=False)

print(f"✅ Saved CSV  : {output_csv}")
print(f"✅ Saved Excel: {output_excel}")

# Also download locally
files.download(output_csv)
